# HERE FOR REPRODUCIBILITY, SEE `Data/`

## Setup

1. Go [here](https://api.census.gov/data/key_signup.html) to request a Census API Key
2. Put in "William & Mary" for the `Organization Name` and your school email for `Email Address`
3. Once you get the email, activate your key and copy it

## Installs/Imports

In [1]:
# %pip install gtfs-kit
# %pip install cenpy

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from getpass import getpass

import cenpy
import gtfs_kit
import geopandas as gpd

C:\Users\vaugh\AppData\Roaming\Python\Python313\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


---

In [3]:
api_key = getpass("Paste your API key -> ")

## Car Ownership
Visit the [transportation topic](https://data.census.gov/all?q=transportation) on USCB's Data Explorer

`B08201 | Household Size by Vehicles Available` is what we need 

The method below requires an API shortcode, you can find them in the citations of the tables:

*U.S. Census Bureau. "Household Size by Vehicles Available." American Community Survey, ACS 1-Year Estimates Detailed Tables, Table B08201, https://data.census.gov/table/ACSDT1Y2025.B08201?q=transportation. Accessed on 23 Nov 2026.*

"ACSDT1Y2025" is the shortcode we need, we'll filter for the actual table right after:

In [4]:
acs = cenpy.remote.APIConnection(api_name='ACSDT1Y2025',apikey=api_key)

In [5]:
acs

Connection to ACS 1-Year Detailed Tables (ID: https://api.census.gov/data/id/ACSDT1Y2025)

In [6]:
# Checking where we can possibly filter for 'B08201'
# Looks like the 'group' column
acs.variables

,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
B24022_060E,Estimate!!Total:!!Female:!!Service occupations...,Sex by Occupation and Median Earnings in the P...,int,B24022,0,NaN,NaN,"B24022_060EA,B24022_060M,B24022_060MA",NaN
B19001B_014E,"Estimate!!Total:!!$100,000 to $124,999",Household Income in the Past 12 Months (in 202...,int,B19001B,0,NaN,NaN,"B19001B_014EA,B19001B_014M,B19001B_014MA",NaN
...,...,...,...,...,...,...,...,...,...
B20005I_071E,"Estimate!!Total:!!Female:!!Worked full-time, y...",Sex by Work Experience in the Past 12 Months b...,int,B20005I,0,NaN,NaN,"B20005I_071EA,B20005I_071M,B20005I_071MA",NaN
B08113_054E,Estimate!!Total:!!Worked from home:!!Speak oth...,Means of Transportation to Work by Language Sp...,int,B08113,0,NaN,NaN,"B08113_054EA,B08113_054M,B08113_054MA",NaN
B06009_006E,Estimate!!Total:!!Graduate or professional degree,Place of Birth by Educational Attainment in th...,int,B06009,0,NaN,NaN,"B06009_006EA,B06009_006M,B06009_006MA",NaN
B05009_012E,Estimate!!Total:!!Under 6 years:!!Living with ...,Age and Nativity of Own Children Under 18 Year...,int,B05009,0,NaN,NaN,"B05009_012EA,B05009_012M,B05009_012MA",NaN


In [7]:
cols = pd.DataFrame(acs.variables[acs.variables['group']=="B08201"]['label'])
cols

,label
B08201_007E,Estimate!!Total:!!1-person household:
B08201_008E,Estimate!!Total:!!1-person household:!!No vehi...
B08201_005E,Estimate!!Total:!!3 vehicles available
B08201_006E,Estimate!!Total:!!4 or more vehicles available
B08201_003E,Estimate!!Total:!!1 vehicle available
B08201_004E,Estimate!!Total:!!2 vehicles available
B08201_001E,Estimate!!Total:
B08201_002E,Estimate!!Total:!!No vehicle available
B08201_030E,Estimate!!Total:!!4-or-more-person household:!...
B08201_029E,Estimate!!Total:!!4-or-more-person household:!...


In [8]:
#We need to find the state and place codes for Chicago, Philadelphia, and Boston to prepare the query
places = acs.query(cols=["NAME"],
                   geo_unit="place:*",
                   geo_filter={"state": "*"})
places

,NAME,state,place
0,"Auburn city, Alabama",01,03076
1,"Birmingham city, Alabama",01,07000
2,"Dothan city, Alabama",01,21184
3,"Hoover city, Alabama",01,35896
4,"Huntsville city, Alabama",01,37000
...,...,...,...
667,"Caguas zona urbana, Puerto Rico",72,10334
668,"Carolina zona urbana, Puerto Rico",72,14290
669,"Guaynabo zona urbana, Puerto Rico",72,32522
670,"Ponce zona urbana, Puerto Rico",72,63820


In [9]:
city_names = ['Chicago city, Illinois','Philadelphia city, Pennsylvania','Boston city, Massachusetts']
places.set_index("NAME").loc[city_names]

,state,place
NAME,,
"Chicago city, Illinois",17,14000
"Philadelphia city, Pennsylvania",42,60000
"Boston city, Massachusetts",25,07000


In [10]:
cities = {
    "Chicago": ("17","14000"),
    "Philadelphia":("42","60000"),
    "Boston":("25","07000")
}
table = cols.copy()
for city, (state,place) in cities.items():
    result = acs.query(
        cols = cols.index.tolist(),
        geo_unit=f"place:{place}",
        geo_filter={"state": state}
    )
    table[city] = pd.to_numeric(result.loc[0,cols.index.tolist()])
table = table.sort_index()

In [11]:
vehicle_ownership_2025 = table.loc["B08201_001E":"B08201_006E"]
vehicle_ownership_2025

,label,Chicago,Philadelphia,Boston
B08201_001E,Estimate!!Total:,1183062,710210,291271
B08201_002E,Estimate!!Total:!!No vehicle available,315529,227448,104941
B08201_003E,Estimate!!Total:!!1 vehicle available,540959,316811,124851
B08201_004E,Estimate!!Total:!!2 vehicles available,243470,130088,50153
B08201_005E,Estimate!!Total:!!3 vehicles available,58641,23767,9404
B08201_006E,Estimate!!Total:!!4 or more vehicles available,24463,12096,1922


In [12]:
vehicle_ownership_2025.label = ["Total Households", "No vehicle","1 vehicle","2 vehicles","3 vehicles",'4 or more vehicles']
vehicle_ownership_2025

,label,Chicago,Philadelphia,Boston
B08201_001E,Total Households,1183062,710210,291271
B08201_002E,No vehicle,315529,227448,104941
B08201_003E,1 vehicle,540959,316811,124851
B08201_004E,2 vehicles,243470,130088,50153
B08201_005E,3 vehicles,58641,23767,9404
B08201_006E,4 or more vehicles,24463,12096,1922


In [13]:
DATA_PATH = Path("Data/")
DATA_PATH.mkdir(parents=True,exist_ok=True)
vehicle_ownership_2025.to_csv(DATA_PATH/"vehicle_ownership_2025.csv")


## Commuting
Visit the [commuting topic](https://data.census.gov/table/ACSST1Y2025.S0801?q=commute) on USCB's Data Explorer

`S0801 | Commuting Characteristics by Sex` is what we need (totals are available)

"ACSST1Y2025" is the shortcode we need, we'll filter for the actual table right after:

In [14]:
acs = cenpy.remote.APIConnection(api_name="ACSST1Y2025",apikey=api_key)

In [15]:
acs

Connection to ACS 1-Year Subject Tables (ID: https://api.census.gov/data/id/ACSST1Y2025)

In [16]:
cols = pd.DataFrame(acs.variables[acs.variables['group']=="S0801"]['label'])
cols

,label
S0801_C02_003E,Estimate!!Male!!Workers 16 years and over!!MEA...
S0801_C02_002E,Estimate!!Male!!Workers 16 years and over!!MEA...
S0801_C02_001E,Estimate!!Male!!Workers 16 years and over
S0801_C02_007E,Estimate!!Male!!Workers 16 years and over!!MEA...
S0801_C02_006E,Estimate!!Male!!Workers 16 years and over!!MEA...
...,...
S0801_C03_035E,Estimate!!Female!!Workers 16 years and over wh...
S0801_C03_036E,Estimate!!Female!!Workers 16 years and over wh...
S0801_C03_037E,Estimate!!Female!!Workers 16 years and over wh...
S0801_C03_038E,Estimate!!Female!!Workers 16 years and over wh...


In [17]:
# Now filter for the Total/TRAVEL TIME TO WORK variables
# 
cols = cols[cols['label'].str.contains("TRAVEL") & cols['label'].str.contains("Total")]
cols

,label
S0801_C01_038E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_037E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_039E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_040E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_046E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_045E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_042E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_041E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_044E,Estimate!!Total!!Workers 16 years and over who...
S0801_C01_043E,Estimate!!Total!!Workers 16 years and over who...


In [18]:
table = cols.copy()
for city, (state,place) in cities.items():
    result = acs.query(
        cols = cols.index.tolist(),
        geo_unit=f"place:{place}",
        geo_filter={"state": state}
    )
    table[city] = pd.to_numeric(result.loc[0,cols.index.tolist()])
table = table.sort_index()
table

,label,Chicago,Philadelphia,Boston
S0801_C01_037E,Estimate!!Total!!Workers 16 years and over who...,5.1,5.2,6.4
S0801_C01_038E,Estimate!!Total!!Workers 16 years and over who...,7.8,7.6,6.2
S0801_C01_039E,Estimate!!Total!!Workers 16 years and over who...,10.2,11.9,11.4
S0801_C01_040E,Estimate!!Total!!Workers 16 years and over who...,13.1,14.4,13.2
S0801_C01_041E,Estimate!!Total!!Workers 16 years and over who...,6.9,6.8,7.9
S0801_C01_042E,Estimate!!Total!!Workers 16 years and over who...,17.2,17.6,19.6
S0801_C01_043E,Estimate!!Total!!Workers 16 years and over who...,11.4,10.0,11.2
S0801_C01_044E,Estimate!!Total!!Workers 16 years and over who...,13.3,11.8,12.6
S0801_C01_045E,Estimate!!Total!!Workers 16 years and over who...,15.0,14.7,11.5
S0801_C01_046E,Estimate!!Total!!Workers 16 years and over who...,33.6,33.5,32.2


In [19]:
table.label = ["Less than 10 minutes (%)",
               "10 to 14 minutes (%)",
               "15 to 19 minutes (%)",
               "20 to 24 minutes (%)",
               "25 to 29 minutes (%)",
               "30 to 34 minutes (%)",
               "35 to 44 minutes (%)",
               "45 to 59 minutes (%)",
               "60 or more minutes (%)",
               "Mean travel time to work (minutes)"]
table

,label,Chicago,Philadelphia,Boston
S0801_C01_037E,Less than 10 minutes (%),5.1,5.2,6.4
S0801_C01_038E,10 to 14 minutes (%),7.8,7.6,6.2
S0801_C01_039E,15 to 19 minutes (%),10.2,11.9,11.4
S0801_C01_040E,20 to 24 minutes (%),13.1,14.4,13.2
S0801_C01_041E,25 to 29 minutes (%),6.9,6.8,7.9
S0801_C01_042E,30 to 34 minutes (%),17.2,17.6,19.6
S0801_C01_043E,35 to 44 minutes (%),11.4,10.0,11.2
S0801_C01_044E,45 to 59 minutes (%),13.3,11.8,12.6
S0801_C01_045E,60 or more minutes (%),15.0,14.7,11.5
S0801_C01_046E,Mean travel time to work (minutes),33.6,33.5,32.2


In [20]:
table.to_csv(DATA_PATH/"commutes_2025.csv")

## Transit Accessibility
The `gtfs/` folder contains transit feeds for December 2025 under the CTA, MBTA, and SEPTA.

In [21]:
# Reference: https://stackoverflow.com/questions/76369026/how-to-access-us-census-tiger-shapefiles-in-geopandas
boundary_rows = []

for city, (state,place) in cities.items():
    url = f"https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_{state}_place.zip"
    places = gpd.read_file(url)
    row = places.loc[places["PLACEFP"] == place, ["ALAND","geometry"]].copy()
    row["city"] = city
    boundary_rows.append(row)

boundaries = gpd.GeoDataFrame(pd.concat(boundary_rows,ignore_index=True))
#GTFS stop points are in EPSG:4326
boundaries = boundaries.to_crs("EPSG:4326")
boundaries["area_sq_mi"] = boundaries["ALAND"] / (1609.344 ** 2)
boundaries


,ALAND,geometry,city,area_sq_mi
0,589864490,"POLYGON ((-87.9401 42.00079, -87.93993 42.0007...",Chicago,227.747953
1,347900745,"POLYGON ((-75.28027 39.97496, -75.27951 39.975...",Philadelphia,134.325229
2,125203166,"POLYGON ((-71.19123 42.283, -71.19119 42.28304...",Boston,48.341213


In [22]:
gtfs_zips = {"Chicago": Path("gtfs/cta.zip"),
             "Philadelphia": Path("gtfs/septa.zip"),
             "Boston": Path("gtfs/mbta.zip")}

service_date = "20251217" #Some random date in December, this was a Wednesday so likely many stops
route_types = [0,1,3,11] #Light rail, subway, bus, trolleybus
city_stop_layers = []

for city, path in gtfs_zips.items():
    feed = gtfs_kit.read_feed(path, dist_units="mi")

    #These are all the routes for the modes we're interested in
    routes = feed.routes.loc[feed.routes["route_type"].isin(route_types)]

    #These are the stop points visited by those routes on 2025-12-17 
    stops = feed.get_stops(date=service_date,
                           route_ids=routes["route_id"].tolist(),
                           as_gdf=True)

    #We want to keep points inside or on the city boundary
    boundary = boundaries.loc[boundaries["city"] == city, ["city", "geometry"]]
    inside = gpd.sjoin(stops, boundary, how="inner", predicate="intersects")
    inside = inside.drop_duplicates(subset="stop_id")

    city_stop_layers.append(inside[["city","stop_id","geometry"]])

city_stops = gpd.GeoDataFrame(pd.concat(city_stop_layers, ignore_index=True))
city_stops.groupby("city").size()


city
Boston           1937
Chicago         10095
Philadelphia     8076
dtype: int64

In [23]:
# Read back the already-prepared CSVs
commutes = pd.read_csv(DATA_PATH/"commutes_2025.csv").set_index(['label'])
vehicle_ownership = pd.read_csv(DATA_PATH/"vehicle_ownership_2025.csv").set_index(['label'])

In [24]:
commutes

,Unnamed: 0,Chicago,Philadelphia,Boston
label,,,,
Less than 10 minutes (%),S0801_C01_037E,5.1,5.2,6.4
10 to 14 minutes (%),S0801_C01_038E,7.8,7.6,6.2
15 to 19 minutes (%),S0801_C01_039E,10.2,11.9,11.4
20 to 24 minutes (%),S0801_C01_040E,13.1,14.4,13.2
25 to 29 minutes (%),S0801_C01_041E,6.9,6.8,7.9
30 to 34 minutes (%),S0801_C01_042E,17.2,17.6,19.6
35 to 44 minutes (%),S0801_C01_043E,11.4,10.0,11.2
45 to 59 minutes (%),S0801_C01_044E,13.3,11.8,12.6
60 or more minutes (%),S0801_C01_045E,15.0,14.7,11.5


In [25]:
vehicle_ownership

,Unnamed: 0,Chicago,Philadelphia,Boston
label,,,,
Total Households,B08201_001E,1183062,710210,291271
No vehicle,B08201_002E,315529,227448,104941
1 vehicle,B08201_003E,540959,316811,124851
2 vehicles,B08201_004E,243470,130088,50153
3 vehicles,B08201_005E,58641,23767,9404
4 or more vehicles,B08201_006E,24463,12096,1922


In [26]:
comparison = boundaries.set_index("city")[["area_sq_mi"]].copy()
comparison["active_stops"] = city_stops.groupby("city").size()
comparison["stops_per_sq_mi"] = comparison["active_stops"] / comparison["area_sq_mi"]

city_names = ['Chicago','Philadelphia','Boston']
no_vehicle = pd.to_numeric(vehicle_ownership.loc["No vehicle", city_names])
households = pd.to_numeric(vehicle_ownership.loc["Total Households", city_names])

#These two metrics aren't needed rn, but are helpful to go straight into visualizing
comparison["pct_no_vehicle"] = (no_vehicle/households)*100
comparison["mean_commute_mins"] = pd.to_numeric(commutes.loc["Mean travel time to work (minutes)", city_names])
comparison

,area_sq_mi,active_stops,stops_per_sq_mi,pct_no_vehicle,mean_commute_mins
city,,,,,
Chicago,227.747953,10095,44.325316,26.670538,33.6
Philadelphia,134.325229,8076,60.122734,32.025457,33.5
Boston,48.341213,1937,40.069330,36.028647,32.2


In [27]:
comparison.to_csv(DATA_PATH/"stop_density_etc_metrics.csv")